# Descriptive Analysis: Identify Bonds to Focus On

Most municipal bonds trade infrequently. For modeling work, focus on the small subset that trades often enough to support time-series analysis.

This notebook:
1. Computes per-CUSIP activity statistics (`NUM_TRADES`, `NUM_DATES_TRADED`, `FRACTION_DAYS_TRADED`).
2. Visualizes the distribution.
3. Selects an active universe by liquidity thresholds.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt

client = bigquery.Client(project='nyu-datasets')
TABLE = 'nyu-datasets.munibonds.trades_typed'

## Compute per-CUSIP activity statistics

Aggregates each bond's trading footprint:
- `NUM_TRADES`: total trades observed
- `D_TRADES`: dealer-to-dealer trades only (`TRADE_TYPE_INDICATOR = 'D'`)
- `NUM_DATES_TRADED`: distinct calendar days with trading
- `DAYS_AVAILABLE`: span between first and last observed trade
- `FRACTION_DAYS_TRADED`: `NUM_DATES_TRADED / DAYS_AVAILABLE` (1.0 = traded every day)

In [ ]:
STATS_QUERY = f"""
SELECT
  CUSIP,
  ANY_VALUE(SECURITY_DESCRIPTION) AS SECURITY_DESCRIPTION,
  MIN(MATURITY_DATE) AS MATURITY_DATE,
  MIN(DATED_DATE) AS DATED_DATE,
  MIN(COUPON) AS COUPON,
  COUNT(*) AS NUM_TRADES,
  COUNTIF(TRADE_TYPE_INDICATOR = 'D') AS D_TRADES,
  COUNT(DISTINCT TRADE_DATE) AS NUM_DATES_TRADED,
  DATE_DIFF(MAX(TRADE_DATE), MIN(TRADE_DATE), DAY) + 1 AS DAYS_AVAILABLE,
  SAFE_DIVIDE(
    COUNT(DISTINCT TRADE_DATE),
    DATE_DIFF(MAX(TRADE_DATE), MIN(TRADE_DATE), DAY) + 1
  ) AS FRACTION_DAYS_TRADED
FROM `{TABLE}`
WHERE CUSIP IS NOT NULL AND TRADE_DATE IS NOT NULL
GROUP BY CUSIP
"""

df = client.query(STATS_QUERY).to_dataframe()
print(f'{len(df):,} unique CUSIPs')
df.head()

In [ ]:
df.describe()

In [ ]:
df.quantile(q=[0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.999], numeric_only=True)

## Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# NUM_TRADES histogram (log-y)
df.NUM_TRADES.hist(bins=100, ax=axes[0, 0])
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('NUM_TRADES per CUSIP (log y)')
axes[0, 0].set_xlabel('NUM_TRADES')

# NUM_TRADES CDF, zoomed to <=100
df.NUM_TRADES.hist(bins=10000, cumulative=True, density=True, ax=axes[0, 1])
axes[0, 1].set_xlim(0, 100)
axes[0, 1].set_title('NUM_TRADES CDF (zoom 0-100)')
axes[0, 1].set_xlabel('NUM_TRADES')

# NUM_DATES_TRADED histogram (log-y)
df.NUM_DATES_TRADED.hist(bins=100, ax=axes[1, 0])
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('NUM_DATES_TRADED per CUSIP (log y)')
axes[1, 0].set_xlabel('NUM_DATES_TRADED')

# FRACTION_DAYS_TRADED CDF, zoomed to <=0.2
active = df[df.NUM_DATES_TRADED > 10]
active.FRACTION_DAYS_TRADED.hist(bins=200, cumulative=True, density=True, ax=axes[1, 1])
axes[1, 1].set_xlim(0, 0.2)
axes[1, 1].set_title('FRACTION_DAYS_TRADED CDF (CUSIPs w/ >10 trade days)')
axes[1, 1].set_xlabel('FRACTION_DAYS_TRADED')

plt.tight_layout()
plt.show()

## Active universe selection

Filter to bonds with enough activity to support time-series modeling:
- More than **100** dealer-to-dealer trades
- More than **10** days with trades
- Active on average **at least once every 10 days** (`FRACTION_DAYS_TRADED > 0.1`)

Adjust thresholds via the variables below.

In [ ]:
MIN_D_TRADES = 100
MIN_DATES_TRADED = 10
MIN_FRACTION_DAYS_TRADED = 0.1

universe = df[
    (df.D_TRADES > MIN_D_TRADES)
    & (df.NUM_DATES_TRADED > MIN_DATES_TRADED)
    & (df.FRACTION_DAYS_TRADED > MIN_FRACTION_DAYS_TRADED)
].sort_values('NUM_TRADES', ascending=False)

print(f'Selected {len(universe):,} of {len(df):,} CUSIPs ({len(universe)/len(df):.2%})')
universe.head(20)

## Optional: persist the universe

Export to GCS or write back to BigQuery for downstream use.

In [ ]:
# Example: write to a BigQuery table
# job = client.load_table_from_dataframe(
#     universe, 'nyu-datasets.munibonds.active_universe',
#     job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE'),
# )
# job.result()
# print(f'Wrote {len(universe):,} rows to nyu-datasets.munibonds.active_universe')